# RL Breakout: Training and Evaluation Notebook

Welcome! This notebook provides a complete workflow to train and evaluate a Deep Q-Network (DQN) agent for the Atari game Breakout.

We will cover the following steps:
1.  **Setup & Installation**: Install all necessary libraries.
2.  **Environment Pre-processing**: Create wrappers to prepare the game frames for the neural network.
3.  **Model & Agent Definition**: Define the CNN architecture, Replay Buffer, and the DQN Agent class.
4.  **Training**: Run the main training loop to teach the agent.
5.  **Evaluation**: Load a trained model and watch it play, recording a video of its performance.

## 1. Setup & Installation

First, let's install the required Python packages. If you have already installed them, you can skip this step.

In [ ]:
%pip install tensorflow gymnasium[atari] gymnasium[accept-rom-license] numpy opencv-python imageio-ffmpeg

## 2. Imports and Environment Setup

Now, we import all the necessary modules and define the wrappers for pre-processing the game environment.

In [ ]:
import cv2
import numpy as np
import gymnasium as gym
from collections import deque
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D
from tensorflow.keras.optimizers import Adam
import os
import imageio

### Environment Wrappers

The raw Atari environment returns large, colorful frames. We need to process them to make them suitable for a CNN. Our wrappers will:
1.  Convert frames to grayscale.
2.  Resize frames to 84x84.
3.  Stack 4 consecutive frames together to give the agent a sense of motion.

In [ ]:
class PreprocessFrame(gym.ObservationWrapper):
    def __init__(self, env, shape):
        super(PreprocessFrame, self).__init__(env)
        self.shape = (shape[2], shape[0], shape[1])
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=self.shape, dtype=np.float32)

    def observation(self, obs):
        new_frame = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        resized_screen = cv2.resize(new_frame, self.shape[1:], interpolation=cv2.INTER_AREA)
        new_obs = np.array(resized_screen, dtype=np.uint8).reshape(self.shape)
        new_obs = new_obs / 255.0
        return new_obs

class StackFrames(gym.ObservationWrapper):
    def __init__(self, env, n_steps):
        super(StackFrames, self).__init__(env)
        self.observation_space = gym.spaces.Box(env.observation_space.low.repeat(n_steps, axis=0),
                                               env.observation_space.high.repeat(n_steps, axis=0), dtype=np.float32)
        self.stack = deque(maxlen=n_steps)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        for _ in range(self.stack.maxlen):
            self.stack.append(obs)
        return np.array(self.stack).reshape(self.observation_space.low.shape), info

    def observation(self, obs):
        self.stack.append(obs)
        return np.array(self.stack).reshape(self.observation_space.low.shape)

def make_env(env_name, shape=(84, 84, 1), n_stack=4):
    """Helper function to create and wrap the environment."""
    env = gym.make(env_name)
    env = StackFrames(PreprocessFrame(env, shape), n_stack)
    return env

## 3. Model and Agent Definition

In [ ]:
def build_dqn(learning_rate, n_actions, input_shape):
    """Builds a Deep Q-Network model."""
    model = Sequential([
        Conv2D(32, (8, 8), strides=(4, 4), activation='relu', input_shape=input_shape),
        Conv2D(64, (4, 4), strides=(2, 2), activation='relu'),
        Conv2D(64, (3, 3), strides=(1, 1), activation='relu'),
        Flatten(),
        Dense(512, activation='relu'),
        Dense(n_actions, activation='linear')
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='huber')
    return model

class ReplayBuffer:
    def __init__(self, max_size, input_shape, n_actions):
        self.mem_size = max_size
        self.mem_counter = 0
        self.state_memory = np.zeros((self.mem_size, *input_shape), dtype=np.float32)
        self.new_state_memory = np.zeros((self.mem_size, *input_shape), dtype=np.float32)
        self.action_memory = np.zeros(self.mem_size, dtype=np.int32)
        self.reward_memory = np.zeros(self.mem_size, dtype=np.float32)
        self.terminal_memory = np.zeros(self.mem_size, dtype=np.bool_)

    def store_transition(self, state, action, reward, new_state, done):
        index = self.mem_counter % self.mem_size
        self.state_memory[index] = state
        self.new_state_memory[index] = new_state
        self.action_memory[index] = action
        self.reward_memory[index] = reward
        self.terminal_memory[index] = done
        self.mem_counter += 1

    def sample_buffer(self, batch_size):
        max_mem = min(self.mem_counter, self.mem_size)
        batch = np.random.choice(max_mem, batch_size, replace=False)
        states = self.state_memory[batch]
        new_states = self.new_state_memory[batch]
        actions = self.action_memory[batch]
        rewards = self.reward_memory[batch]
        dones = self.terminal_memory[batch]
        return states, actions, rewards, new_states, dones

class DQNAgent:
    def __init__(self, lr, gamma, n_actions, epsilon, batch_size, input_dims, 
                 epsilon_dec=1e-5, epsilon_end=0.01, mem_size=100000, fname='dqn_model.h5'):
        self.action_space = [i for i in range(n_actions)]
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_dec = epsilon_dec
        self.epsilon_min = epsilon_end
        self.batch_size = batch_size
        self.model_file = fname
        self.memory = ReplayBuffer(mem_size, input_dims, n_actions)
        self.q_eval = build_dqn(lr, n_actions, input_dims)

    def store_transition(self, state, action, reward, new_state, done):
        self.memory.store_transition(state, action, reward, new_state, done)

    def choose_action(self, observation):
        if np.random.random() < self.epsilon:
            action = np.random.choice(self.action_space)
        else:
            state = np.array([observation], copy=False, dtype=np.float32)
            actions = self.q_eval.predict(state, verbose=0)
            action = np.argmax(actions)
        return action

    def learn(self):
        if self.memory.mem_counter < self.batch_size:
            return

        states, actions, rewards, new_states, dones = self.memory.sample_buffer(self.batch_size)

        q_eval = self.q_eval.predict(states, verbose=0)
        q_next = self.q_eval.predict(new_states, verbose=0)

        q_target = np.copy(q_eval)
        batch_index = np.arange(self.batch_size, dtype=np.int32)

        q_target[batch_index, actions] = rewards + self.gamma * np.max(q_next, axis=1) * (1 - dones)

        self.q_eval.train_on_batch(states, q_target)

        self.epsilon = self.epsilon - self.epsilon_dec if self.epsilon > self.epsilon_min else self.epsilon_min
    
    def save_model(self):
        self.q_eval.save(self.model_file)
        
    def load_model(self):
        self.q_eval = tf.keras.models.load_model(self.model_file)

## 4. Training the Agent

Now we'll set up the main training loop. 

**Note**: Training a DQN agent from scratch takes a very long time (hours to days). The number of episodes (`n_games`) is set to a small value for demonstration purposes. For serious training, you should increase this to 5000 or more and consider running this on a machine with a GPU.

In [ ]:
ENV_NAME = 'BreakoutNoFrameskip-v4'

# Create directories for saving models and videos
if not os.path.exists('models'):
    os.makedirs('models')
if not os.path.exists('videos'):
    os.makedirs('videos')

env = make_env(ENV_NAME)

# Hyperparameters
n_games = 500  # For demonstration. Increase for real training.
lr = 0.0001
gamma = 0.99
epsilon = 1.0
epsilon_dec = 1e-5
epsilon_end = 0.1
mem_size = 20000
batch_size = 32
input_dims = env.observation_space.shape
n_actions = env.action_space.n

agent = DQNAgent(lr=lr, gamma=gamma, n_actions=n_actions, epsilon=epsilon,
                 batch_size=batch_size, input_dims=input_dims, mem_size=mem_size,
                 epsilon_dec=epsilon_dec, epsilon_end=epsilon_end,
                 fname='models/breakout_dqn.h5')

scores = []
eps_history = []

print("Starting Training...")

for i in range(n_games):
    done = False
    score = 0
    observation, info = env.reset()
    
    while not done:
        action = agent.choose_action(observation)
        observation_, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        score += reward
        agent.store_transition(observation, action, reward, observation_, done)
        observation = observation_
        agent.learn()
        
    eps_history.append(agent.epsilon)
    scores.append(score)
    
    avg_score = np.mean(scores[-100:])
    print(f'Episode: {i}, Score: {score}, Average Score: {avg_score:.2f}, Epsilon: {agent.epsilon:.4f}')
    
    # Save model periodically
    if i % 50 == 0 and i > 0:
        print(f'Saving model at episode {i}...')
        agent.save_model()

print("Training finished.")
agent.save_model()
print("Final model saved.")

## 5. Evaluation

After training, let's evaluate the agent's performance. We'll load the saved model and run it in the environment with rendering enabled. We will also record a video of the agent playing.

In [ ]:
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML
from base64 import b64encode

# Load the trained agent
print("Loading trained model for evaluation...")
eval_agent = DQNAgent(lr=lr, gamma=gamma, n_actions=n_actions, epsilon=0.0, # Epsilon is 0 for eval
                      batch_size=batch_size, input_dims=input_dims, mem_size=mem_size,
                      epsilon_dec=epsilon_dec, epsilon_end=epsilon_end,
                      fname='models/breakout_dqn.h5')
eval_agent.load_model()

# Create a new environment for evaluation and video recording
video_env = make_env(ENV_NAME)
video_env = RecordVideo(video_env, 'videos/', name_prefix='dqn-breakout-eval')

print("Running evaluation...")
observation, info = video_env.reset()
done = False
total_reward = 0

while not done:
    action = eval_agent.choose_action(observation)  # Use the agent to choose the best action
    observation, reward, terminated, truncated, info = video_env.step(action)
    done = terminated or truncated
    total_reward += reward

print(f"Evaluation finished. Total Reward: {total_reward}")
video_env.close()

print("Displaying recorded video...")
# Find the latest video file
video_files = sorted([f for f in os.listdir('videos') if f.endswith('.mp4')])
if video_files:
    video_path = os.path.join('videos', video_files[-1])
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'<video controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))
else:
    print("No video file found.")